<a href="https://colab.research.google.com/github/HemanthSagarSR/AI_Lab-/blob/main/week7_FOL_unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re

def parse_expr(s):
    s = s.replace(" ", "")
    if '(' not in s:
        return s
    match = re.match(r"^([A-Za-z0-9_]+)\((.*)\)$", s)
    if not match:
        return s
    pred = match.group(1)
    args_str = match.group(2)

    args = []
    current = []
    depth = 0
    for char in args_str:
        if char == '(':
            depth += 1
            current.append(char)
        elif char == ')':
            depth -= 1
            current.append(char)
        elif char == ',' and depth == 0:
            args.append(parse_expr("".join(current)))
            current = []
        else:
            current.append(char)
    if current:
        args.append(parse_expr("".join(current)))
    return [pred] + args

def to_string(expr):
    if isinstance(expr, str):
        return expr
    return f"{expr}[0]({', '.join(to_string(arg) for arg in expr[1:])})"

def is_variable(x):
    return isinstance(x, str) and x.islower() and len(x) == 1

def occurs_in(var, term):
    if var == term:
        return True
    if isinstance(term, list):
        return any(occurs_in(var, arg) for arg in term[1:])
    return False

def apply_subst(subst, term):
    if not subst:
        return term
    if isinstance(term, str):
        return subst.get(term, term)
    if isinstance(term, list):
        return [term[0]] + [apply_subst(subst, arg) for arg in term[1:]]
    return term

def compose_subst(snew, subst):
    res = {}
    for var, val in subst.items():
        res[var] = apply_subst(snew, val)
    for var, val in snew.items():
        if var not in res:
            res[var] = val
    return res

def unify(psi1, psi2):
    if isinstance(psi1, str) or isinstance(psi2, str):
        if psi1 == psi2:
            return {}
        if is_variable(psi1):
            if occurs_in(psi1, psi2):
                return "FAILURE"
            return {psi1: psi2}
        if is_variable(psi2):
            if occurs_in(psi2, psi1):
                return "FAILURE"
            return {psi2: psi1}
        return "FAILURE"

    if psi1[0] != psi2[0]:
        return "FAILURE"

    if len(psi1) != len(psi2):
        return "FAILURE"

    subst = {}

    for i in range(1, len(psi1)):
        s = unify(psi1[i], psi2[i])
        if s == "FAILURE":
            return "FAILURE"
        if s != {}:
            psi1 = [psi1[0]] + [apply_subst(s, arg) for arg in psi1[1:]]
            psi2 = [psi2[0]] + [apply_subst(s, arg) for arg in psi2[1:]]
            subst = compose_subst(s, subst)

    return subst

def main():
    print("--- First-Order Logic Unification Program ---")
    str1 = input("Enter Expression 1: ").strip()
    str2 = input("Enter Expression 2: ").strip()

    try:
        psi1 = parse_expr(str1)
        psi2 = parse_expr(str2)

        result = unify(psi1, psi2)

        print("\n--- Result ---")
        if result == "FAILURE":
            print(" Unification Failed: No unifier exists.")
        elif result == {}:
            print(" Unification Successful: The expressions are already identical (Substitution Set is NIL).")
        else:
            print(" Unification Successful!")
            print("Most General Unifier (MGU):")
            for var, val in result.items():
                print(f"  {var} = {to_string(val)}")

            unified_expr = to_string(apply_subst(result, psi1))
            print(f"Unified Expression: {unified_expr}")

    except Exception as e:
        print(f"Parsing error: {e}")

if __name__ == "__main__":
    main()


--- First-Order Logic Unification Program ---
Enter Expression 1:  Eats(x, Apple)
Enter Expression 2: Eats(Riya, y)

--- Result ---
 Unification Successful!
Most General Unifier (MGU):
  x = Riya
  y = Apple
Unified Expression: ['Eats', 'Riya', 'Apple'][0](Riya, Apple)
